# Music Annotation Project V9.0A
## YOLO Mask Authority Transfer Audit

**Runtime:** CPU if Notebook 06 draft masks/confidence maps already exist. This notebook does not retrain YOLO.

This revision is fail-closed and follows the executable project lineage rather than assuming a generic V8.8 folder. It tests one change only: the source of the annotation/intervention mask presented to the correction path.

Mask arms:
1. existing reviewed/corrected mask, when resolved through the Notebook 07 audit;
2. raw Notebook 06 YOLO draft mask;
3. confidence-gated subsets of that same draft proposal;
4. reviewer-corrected YOLO mask, which is the same reviewed arm when it exists.

**Boundaries**
- Raw YOLO remains proposal evidence, not automatic authority.
- Annotation confidence and correction safety remain separate.
- Clean truth is prohibited until outputs are written and locked.
- No V8.8 policy threshold, reconstruction rule, detector, or safety rule is retuned here.
- The notebook resolves V8.8 through the completed Notebook 57R1 release-control lineage and recorded upstream paths. It does not search for `/v8_8/`.
- If the required registered-replay executable contract cannot be resolved exactly, the run stops rather than substituting guessed code.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib


## 1. Configuration


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, sys, platform
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
PILOT_ROOT=ROOT/'production_pilot'
INCOMING=PILOT_ROOT/'incoming'
DRAFT_AUDIT_DIR=ROOT/'human_in_the_middle'/'draft_audit'
REVIEW_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'

# These roots come from the executable V8.8 notebooks.
N53_ROOT=ROOT/'observed_print_classifier_v8_8_notebook53R3'
N54_ROOT=ROOT/'observed_print_classifier_v8_8_notebook54'
N55_ROOT=ROOT/'observed_print_classifier_v8_8_notebook55'
N56_ROOT=ROOT/'observed_print_classifier_v8_8_notebook56'
N57_ROOT=ROOT/'v8_8_registered_replay_release_control'

# Exact-run overrides are preferred if more than one complete candidate exists.
N57_RUN_OVERRIDE=None
N56_RUN_OVERRIDE=None
N55_RUN_OVERRIDE=None
N54_RUN_OVERRIDE=None
N53_RUN_OVERRIDE=None

# The registered replay executable function/module name is deliberately not guessed.
# V9.0A resolves it from the selected release record or stops with a precise message.

CONFIDENCE_THRESHOLDS=[0.90,0.95,0.98,0.995]
SEED=42
MAX_VISUAL_CASES=30
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v9_0a_yolo_mask_authority_transfer'
OUT=ROOT/'observed_print_classifier_v9_0a_mask_authority_transfer'/RUN
INPUTS=OUT/'inputs'; MASKS=OUT/'mask_arms'; OUTPUTS=OUT/'outputs'; TABLES=OUT/'tables'; AUDIT=OUT/'audit'; VISUAL=OUT/'visual_audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,MASKS,OUTPUTS,TABLES,AUDIT,VISUAL,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
print('Output:',OUT)


## 2. Helpers and exact V8.8 lineage resolution

Notebook 53R3 explicitly uses the project root plus named Notebook 45/46/50/51 roots, and writes to `observed_print_classifier_v8_8_notebook53R3`. It also distinguishes historical `REGISTERED_REPLAY` from the commissioned REMOVE-only policy. V9.0A therefore does not infer a monolithic V8.8 directory or silently treat the Notebook 51R2 historical quirk as an operational policy.


In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def object_hash(v): return hashlib.sha256(json.dumps(v,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()

def read_json(path):
    p=Path(path)
    if not p.exists(): raise RuntimeError(f'Missing required JSON: {p}')
    return json.loads(p.read_text())

def complete_dirs(root, required):
    root=Path(root); hits=[]
    if root.exists():
        for p in root.rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return sorted(hits,key=lambda p:p.stat().st_mtime)

def resolve_one(root,required,override,label):
    if override is not None:
        p=Path(override); missing=[r for r in required if not (p/r).exists()]
        if missing: raise RuntimeError(f'{label} override incomplete: {missing}')
        return p
    hits=complete_dirs(root,required)
    if not hits: raise RuntimeError(f'No complete {label} run found below {root}.')
    if len(hits)>1:
        print(f'{label}: {len(hits)} complete candidates found; choosing most recently modified. Set the override to pin an exact run.')
    return hits[-1]

# Notebook 57R1 is the downstream release-control record. Required names here are from its published contract.
N57=resolve_one(N57_ROOT,['release_candidate'],N57_RUN_OVERRIDE,'Notebook57R1 release-control')
print('Notebook57R1:',N57)

# Inspect JSON records in the selected N57 run for recorded upstream paths.
def json_records(root):
    out=[]
    for p in Path(root).rglob('*.json'):
        try: out.append((p,json.loads(p.read_text())))
        except Exception: pass
    return out

def find_recorded_path(records, tokens):
    found=[]
    def walk(v):
        if isinstance(v,dict):
            for x in v.values(): walk(x)
        elif isinstance(v,list):
            for x in v: walk(x)
        elif isinstance(v,str):
            low=v.lower()
            if all(t.lower() in low for t in tokens): found.append(v)
    for _,r in records: walk(r)
    uniq=[]
    for x in found:
        if x not in uniq: uniq.append(x)
    return uniq

n57_records=json_records(N57)
recorded56=find_recorded_path(n57_records,['notebook56'])+find_recorded_path(n57_records,['v8_8e'])
if N56_RUN_OVERRIDE: N56=Path(N56_RUN_OVERRIDE)
elif recorded56:
    existing=[Path(x) for x in recorded56 if Path(x).exists()]
    N56=existing[0] if len(existing)==1 else None
else: N56=None
if N56 is None:
    candidates=complete_dirs(N56_ROOT,['pilot_manifest.json'])
    if len(candidates)==1: N56=candidates[0]
    else: raise RuntimeError('Could not uniquely resolve Notebook56 from Notebook57R1 records. Set N56_RUN_OVERRIDE to the exact completed Notebook56 run.')
print('Notebook56:',N56)

# Resolve 55/54/53 from the recorded JSON lineage when possible; fail on ambiguity rather than inventing a run.
records56=json_records(N56)
def resolve_recorded_or_unique(records,tokens,root,required,override,label):
    if override: return Path(override)
    vals=find_recorded_path(records,tokens); existing=[]
    for x in vals:
        p=Path(x)
        if p.exists() and p not in existing: existing.append(p)
    if len(existing)==1: return existing[0]
    hits=complete_dirs(root,required)
    if len(hits)==1: return hits[0]
    raise RuntimeError(f'Could not uniquely resolve {label}. Set its exact override. Recorded candidates={existing}; complete candidates={hits}')

N55=resolve_recorded_or_unique(records56,['notebook55'],N55_ROOT,[],N55_RUN_OVERRIDE,'Notebook55')
records55=json_records(N55)
N54=resolve_recorded_or_unique(records55,['notebook54'],N54_ROOT,[],N54_RUN_OVERRIDE,'Notebook54')
N53=resolve_recorded_or_unique(records55,['notebook53'],N53_ROOT,['stage_f_verdict.json','experiment_identity.json'],N53_RUN_OVERRIDE,'Notebook53R3')
print('Notebook55:',N55); print('Notebook54:',N54); print('Notebook53R3:',N53)

# Guard the known 53R3 lineage distinction.
n53id=read_json(N53/'experiment_identity.json')
if n53id.get('mode')!='REGISTERED_REPLAY':
    raise RuntimeError('V9.0A expected the Stage-F REGISTERED_REPLAY identity for historical compatibility evidence; selected Notebook53R3 has another mode.')
if n53id.get('operational_safe_protect_authorised') is True:
    raise RuntimeError('Unexpected Notebook53R3 identity: SAFE_PROTECT must not be silently authorised.')


## 3. Resolve the registered-replay executable contract from the release candidate

V9.0A must use the released runnable contract, not a newly written approximation. The release candidate is scanned only for explicit Python entry points. If it does not expose a supported page-correction API, the notebook stops and asks for the exact release entry point rather than guessing.


In [ ]:
import importlib.util, inspect
py_files=sorted((N57/'release_candidate').rglob('*.py'))
if not py_files:
    raise RuntimeError('Notebook57R1 release_candidate contains no Python module. V9.0A cannot safely infer how to execute the registered replay. Provide the exact released executable entry point or adapter.')

SUPPORTED_ENTRYPOINTS=['run_registered_replay_page','correct_page_registered_replay','run_v88_page']
resolved=[]
for p in py_files:
    try:
        spec=importlib.util.spec_from_file_location('candidate_'+hashlib.md5(str(p).encode()).hexdigest(),p)
        mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
        for name in SUPPORTED_ENTRYPOINTS:
            if hasattr(mod,name): resolved.append((p,mod,name))
    except Exception:
        continue
if len(resolved)!=1:
    raise RuntimeError('Could not uniquely resolve a supported registered-replay page entry point from Notebook57R1 release_candidate. Found: '+str([(str(p),n) for p,_,n in resolved])+'. Add a small explicit adapter to the release candidate or set the notebook to the documented entry point. V9.0A will not guess.')
ENGINE_PATH,ENGINE_MODULE,ENGINE_NAME=resolved[0]; ENGINE=getattr(ENGINE_MODULE,ENGINE_NAME)
print('Registered replay engine:',ENGINE_PATH,ENGINE_NAME)
print('Signature:',inspect.signature(ENGINE))
ENGINE_SHA256=sha256_file(ENGINE_PATH)


## 4. Resolve Notebook 06 draft and Notebook 07 reviewed evidence by audit record

The pilot lineage already resolves HITL material by source hash. V9.0A uses the same principle. It does not assume that filename stems alone establish mask authority.


In [ ]:
def md5_file(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def audits(folder):
    rows=[]
    for p in sorted(Path(folder).glob('*.json')):
        try:
            r=json.loads(p.read_text()); r['_audit_path']=str(p); rows.append(r)
        except Exception as e: raise RuntimeError(f'Unreadable audit {p}: {e}')
    return rows
DRAFT_AUDITS=audits(DRAFT_AUDIT_DIR); REVIEW_AUDITS=audits(REVIEW_AUDIT_DIR)

def reqpath(record,key):
    raw=record.get(key)
    if not raw: raise RuntimeError(f'Audit {record.get("_audit_path")} lacks {key}')
    p=Path(raw)
    if not p.exists(): raise RuntimeError(f'Audit {record.get("_audit_path")} references missing {key}: {p}')
    return p

def matches(records,h): return [r for r in records if str(r.get('source_md5','')).lower()==h.lower()]

sources=sorted([p for p in INCOMING.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED]) if INCOMING.exists() else []
if not sources: raise RuntimeError(f'No V9.0A source pages found in {INCOMING}. Use the same production-like incoming pages as the V8.8 pilot/release lineage, or set INCOMING deliberately before running.')
resolved=[]
for source in sources:
    h=md5_file(source); da=matches(DRAFT_AUDITS,h); ra=matches(REVIEW_AUDITS,h)
    if len(da)!=1: raise RuntimeError(f'{source.name}: expected exactly one Notebook06 draft audit by source_md5; found {len(da)}')
    if len(ra)>1: raise RuntimeError(f'{source.name}: multiple Notebook07 review audits by source_md5')
    draft=da[0]; draft_mask=reqpath(draft,'draft_mask'); conf=reqpath(draft,'confidence_map')
    review=ra[0] if ra else None
    observed=source; reviewed_mask=None
    if review is not None:
        observed=reqpath(review,'aligned_output'); reviewed_mask=reqpath(review,'corrected_mask_output')
    resolved.append({'page':source.stem,'source':str(source),'source_md5':h,'observed':str(observed),'draft_mask':str(draft_mask),'confidence_map':str(conf),'reviewed_mask':str(reviewed_mask) if reviewed_mask else None,'draft_audit':draft['_audit_path'],'review_audit':review['_audit_path'] if review else None})
cohort=pd.DataFrame(resolved)
if cohort.reviewed_mask.isna().any():
    miss=cohort.loc[cohort.reviewed_mask.isna(),'page'].tolist(); raise RuntimeError(f'Paired authority-transfer audit requires a reviewed comparator for every page. Missing reviewed masks: {miss}')
cohort.to_csv(INPUTS/'paired_input_resolution.csv',index=False)
print('Paired pages:',len(cohort)); display(cohort[['page','draft_mask','confidence_map','reviewed_mask']])


## 5. Build mask arms without changing detector output

Confidence-gated arms are strict subsets of the saved Notebook 06 draft proposal. No morphology, dilation, merge heuristic or new detector inference is introduced here. Any dilation required by the correction engine remains the engine's own frozen contract.


In [ ]:
def binary(a): return np.where(a>0,255,0).astype(np.uint8)
def metrics(ref,cand):
    r=ref>0; c=cand>0; inter=int((r&c).sum()); union=int((r|c).sum())
    return {'reviewed_pixels':int(r.sum()),'candidate_pixels':int(c.sum()),'intersection':inter,'union':union,'iou':inter/max(union,1),'overreach_pixels':int((c&~r).sum()),'missed_reviewed_pixels':int((r&~c).sum()),'xor_pixels':int((r^c).sum())}
page_data={}; rows=[]
for _,r in cohort.iterrows():
    obs=cv2.imread(r.observed,0); raw=binary(cv2.imread(r.draft_mask,0)); reviewed=binary(cv2.imread(r.reviewed_mask,0)); conf=np.load(r.confidence_map).astype(np.float32)
    if obs is None or raw is None or reviewed is None or not (obs.shape==raw.shape==reviewed.shape==conf.shape): raise RuntimeError(f'{r.page}: input shape/readability mismatch')
    arms={'reviewed':reviewed,'raw_yolo':raw}
    for t in CONFIDENCE_THRESHOLDS: arms[f'yolo_conf_{t:.3f}']=np.where((raw>0)&(conf>=t),255,0).astype(np.uint8)
    page_data[r.page]={'observed':obs,'confidence':conf,'arms':arms}
    d=MASKS/r.page; d.mkdir(parents=True,exist_ok=False)
    for arm,m in arms.items():
        p=d/f'{arm}.png'; cv2.imwrite(str(p),m); q=metrics(reviewed,m); q.update({'page':r.page,'arm':arm,'mask_sha256':sha256_file(p)}); rows.append(q)
mask_df=pd.DataFrame(rows); mask_df.to_csv(TABLES/'mask_arm_metrics.csv',index=False)
display(mask_df.groupby('arm')[['iou','overreach_pixels','missed_reviewed_pixels','xor_pixels']].mean())


## 6. Lock inputs before correction


In [ ]:
lock={'created_utc':datetime.now(timezone.utc).isoformat(),'notebook57R1':str(N57),'notebook56':str(N56),'notebook55':str(N55),'notebook54':str(N54),'notebook53R3':str(N53),'registered_replay_engine':str(ENGINE_PATH),'registered_replay_engine_sha256':ENGINE_SHA256,'cohort_sha256':sha256_file(INPUTS/'paired_input_resolution.csv'),'mask_metrics_sha256':sha256_file(TABLES/'mask_arm_metrics.csv'),'confidence_thresholds':CONFIDENCE_THRESHOLDS}
(INPUTS/'v9_0a_input_lock.json').write_text(json.dumps(lock,indent=2))
print('Input lock:',object_hash(lock))


## 7. Execute the released correction engine on each mask arm

The adapter contract is checked at runtime. V9.0A passes only observed image plus authoritative seed mask and a per-arm work directory. If the release API requires something else, execution stops. That is an integration question to resolve explicitly, not a reason to silently emulate V8.8.


In [ ]:
import inspect
sig=inspect.signature(ENGINE)
params=list(sig.parameters)
expected=[['observed','authoritative_mask','work_dir'],['observed','seed_mask','work_dir']]
if params[:3] not in expected:
    raise RuntimeError(f'Released engine signature {params} is not one of the supported explicit contracts {expected}. Add/document an adapter rather than altering V9.0A assumptions.')
output_rows=[]; output_runtime={}
for page,z in page_data.items():
    output_runtime[page]={}
    for arm,seed in z['arms'].items():
        work=OUTPUTS/page/arm; work.mkdir(parents=True,exist_ok=False)
        result=ENGINE(z['observed'].copy(),seed.copy(),work)
        if not isinstance(result,dict) or 'corrected' not in result: raise RuntimeError(f'{page}/{arm}: engine must return dict containing corrected')
        corrected=np.asarray(result['corrected'])
        if corrected.shape!=z['observed'].shape: raise RuntimeError(f'{page}/{arm}: corrected shape mismatch')
        # Engine may prepare/dilate the seed internally. If it returns prepared_mask, use it for the authority-boundary assertion.
        prepared=np.asarray(result.get('prepared_mask',seed))>0
        if prepared.shape!=corrected.shape: raise RuntimeError(f'{page}/{arm}: prepared-mask shape mismatch')
        outside=int(np.sum(corrected[~prepared]!=z['observed'][~prepared]))
        if outside: raise RuntimeError(f'{page}/{arm}: outside-authority drift={outside}')
        cv2.imwrite(str(work/'corrected.png'),corrected)
        output_runtime[page][arm]={'corrected':corrected,'prepared_mask':prepared}
        output_rows.append({'page':page,'arm':arm,'outside_authority_drift':outside,'corrected_sha256':sha256_file(work/'corrected.png'),'prepared_mask_pixels':int(prepared.sum())})
output_df=pd.DataFrame(output_rows); output_df.to_csv(AUDIT/'correction_output_integrity.csv',index=False)
print('Completed arm runs:',len(output_df))


## 8. Prediction/output lock before clean truth or comparator-effect scoring


In [ ]:
prediction_lock={'created_utc':datetime.now(timezone.utc).isoformat(),'input_lock_sha256':sha256_file(INPUTS/'v9_0a_input_lock.json'),'output_integrity_sha256':sha256_file(AUDIT/'correction_output_integrity.csv'),'outputs':output_df[['page','arm','corrected_sha256','outside_authority_drift']].sort_values(['page','arm']).to_dict('records')}
(AUDIT/'prediction_lock.json').write_text(json.dumps(prediction_lock,indent=2))
print('Prediction lock:',object_hash(prediction_lock))


## 9. Paired downstream effect of changing mask authority


In [ ]:
paired=[]
for page,arms in output_runtime.items():
    ref=arms['reviewed']['corrected']
    for arm,q in arms.items():
        if arm=='reviewed': continue
        diff=q['corrected']!=ref
        paired.append({'page':page,'arm':arm,'corrected_xor_vs_reviewed':int(diff.sum()),'candidate_changed_vs_observed':int((q['corrected']!=page_data[page]['observed']).sum()),'reviewed_changed_vs_observed':int((ref!=page_data[page]['observed']).sum())})
paired_df=pd.DataFrame(paired); paired_df.to_csv(TABLES/'paired_downstream_effect.csv',index=False); display(paired_df.groupby('arm').mean(numeric_only=True))


## 10. Optional post-lock clean-truth scoring

Clean truth is resolved from the project `source/clean` directory only after the prediction lock. Pages without clean counterparts remain valid for shadow transfer evidence but are marked unscored.


In [ ]:
def image_map(folder): return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def printed_ink(im): return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def tolerant(ref,pred,region,tol=1):
    ref=ref&region; pred=pred&region
    if not ref.sum(): return np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8); rd=cv2.dilate(ref.astype(np.uint8),k)>0; pdil=cv2.dilate(pred.astype(np.uint8),k)>0
    p=np.sum(pred&rd)/max(pred.sum(),1); r=np.sum(ref&pdil)/ref.sum(); return float(2*p*r/max(p+r,1e-12))
clean_map=image_map(ROOT/'source'/'clean'); truth=[]
for page,arms in output_runtime.items():
    cp=clean_map.get(page)
    if cp is None:
        for arm in arms: truth.append({'page':page,'arm':arm,'clean_available':False})
        continue
    clean=cv2.imread(str(cp),0); ink=printed_ink(clean)
    for arm,q in arms.items():
        region=q['prepared_mask']; out=q['corrected']; oi=printed_ink(out)
        truth.append({'page':page,'arm':arm,'clean_available':True,'mask_pixels':int(region.sum()),'ink_f1_inside_authority':tolerant(ink,oi,region,1),'mae_inside_authority':float(np.abs(clean.astype(np.float32)-out.astype(np.float32))[region].mean()) if region.any() else np.nan})
truth_df=pd.DataFrame(truth); truth_df.to_csv(TABLES/'postlock_clean_truth_scores.csv',index=False)
if truth_df.clean_available.any(): display(truth_df[truth_df.clean_available].groupby('arm')[['ink_f1_inside_authority','mae_inside_authority']].mean())


## 11. Reviewer edit-distance evidence

This measures the difference between Notebook 06 draft and Notebook 07 corrected masks. It is direct evidence about the amount and direction of human mask correction, not a performance label inferred from reviewer behaviour.


In [ ]:
edit=[]
for _,r in cohort.iterrows():
    raw=cv2.imread(r.draft_mask,0)>0; rev=cv2.imread(r.reviewed_mask,0)>0
    edit.append({'page':r.page,'draft_pixels':int(raw.sum()),'reviewed_pixels':int(rev.sum()),'reviewer_add_pixels':int((rev&~raw).sum()),'reviewer_erase_pixels':int((raw&~rev).sum()),'mask_xor_pixels':int((raw^rev).sum()),'mask_xor_fraction_union':float((raw^rev).sum()/max((raw|rev).sum(),1))})
edit_df=pd.DataFrame(edit); edit_df.to_csv(TABLES/'reviewer_mask_edit_distance.csv',index=False); display(edit_df.describe())


## 12. Visual audit queue


In [ ]:
evidence=mask_df.merge(paired_df,on=['page','arm'],how='left'); evidence['review_priority']=evidence.overreach_pixels*5+evidence.missed_reviewed_pixels*3+evidence.corrected_xor_vs_reviewed.fillna(0); evidence.to_csv(TABLES/'paired_transfer_evidence.csv',index=False)
queue=evidence[evidence.arm!='reviewed'].sort_values('review_priority',ascending=False); queue.to_csv(TABLES/'visual_review_queue.csv',index=False)
def sheet(page,arm):
    obs=page_data[page]['observed']; reviewed=page_data[page]['arms']['reviewed']>0; candidate=page_data[page]['arms'][arm]>0; ref=output_runtime[page]['reviewed']['corrected']; out=output_runtime[page][arm]['corrected']
    overlay=cv2.cvtColor(obs,cv2.COLOR_GRAY2RGB); overlay[reviewed&candidate]=(0,180,0); overlay[reviewed&~candidate]=(255,165,0); overlay[candidate&~reviewed]=(255,0,0)
    fig,ax=plt.subplots(1,4,figsize=(20,7));
    for a,im,t in zip(ax,[obs,overlay,ref,out],['Observed','Mask overlap / missed / overreach','Reviewed-mask correction',arm+' correction']): a.imshow(im,cmap='gray' if im.ndim==2 else None); a.set_title(t); a.axis('off')
    fig.suptitle(page+' | '+arm); plt.tight_layout(); return fig
for i,r in queue.head(MAX_VISUAL_CASES).reset_index(drop=True).iterrows():
    fig=sheet(r.page,r.arm); fig.savefig(VISUAL/f'{i+1:03d}_{r.page}_{r.arm}.png',dpi=130,bbox_inches='tight'); plt.close(fig)
print('Visual cases:',min(MAX_VISUAL_CASES,len(queue)))


## 13. Fail-closed V9.0A interpretation

This notebook does not choose an automatic mask-authority threshold. It produces paired development/shadow evidence. Any automatic authority rule requires a separately frozen policy and independent validation.


In [ ]:
summary=evidence[evidence.arm!='reviewed'].groupby('arm',as_index=False).agg(pages=('page','nunique'),mean_mask_iou=('iou','mean'),overreach_pixels=('overreach_pixels','sum'),missed_reviewed_pixels=('missed_reviewed_pixels','sum'),corrected_xor_vs_reviewed=('corrected_xor_vs_reviewed','sum')); summary.to_csv(TABLES/'arm_summary.csv',index=False)
verdict={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'9.0A','experiment':'YOLO mask authority transfer audit','development_shadow_only':True,'automatic_mask_authority_authorised':False,'v88_resolved_from_notebook57R1_lineage':True,'registered_replay_engine':str(ENGINE_PATH),'registered_replay_engine_sha256':ENGINE_SHA256,'notebook53R3_mode':n53id.get('mode'),'safe_protect_operationally_authorised':False,'outside_authority_immutability_passed':bool((output_df.outside_authority_drift==0).all()),'prediction_lock_sha256':sha256_file(AUDIT/'prediction_lock.json'),'interpretation':'Evidence quantifies how saved Notebook06 YOLO proposal masks transfer through the exact released registered-replay engine relative to Notebook07 reviewed-mask authority. It does not itself authorise raw YOLO masks.'}
(OUT/'v9_0a_verdict.json').write_text(json.dumps(verdict,indent=2)); display(summary); print(json.dumps(verdict,indent=2))


## 14. Final Upload Package


In [ ]:
import zipfile
required=[OUT/'v9_0a_verdict.json',INPUTS/'paired_input_resolution.csv',INPUTS/'v9_0a_input_lock.json',AUDIT/'prediction_lock.json',AUDIT/'correction_output_integrity.csv',TABLES/'mask_arm_metrics.csv',TABLES/'paired_downstream_effect.csv',TABLES/'postlock_clean_truth_scores.csv',TABLES/'reviewer_mask_edit_distance.csv',TABLES/'paired_transfer_evidence.csv',TABLES/'arm_summary.csv',TABLES/'visual_review_queue.csv']
for p in required:
    if not p.exists(): raise RuntimeError(f'Missing required upload artefact: {p}')
(UPLOAD/'README.txt').write_text('V9.0A YOLO MASK AUTHORITY TRANSFER AUDIT\nDevelopment/shadow evidence only. Raw YOLO masks are not automatically authoritative.\n')
manifest=[]
for src in required+sorted(VISUAL.glob('*.png'))[:MAX_VISUAL_CASES]:
    rel=('visuals/'+src.name) if src.suffix.lower()=='.png' else ('tables/'+src.name if src.parent==TABLES else 'audit/'+src.name if src.parent==AUDIT else 'inputs/'+src.name if src.parent==INPUTS else src.name)
    dst=UPLOAD/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst); manifest.append({'relative_path':str(dst.relative_to(UPLOAD)),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False); (UPLOAD/'upload_file_list.txt').write_text('\n'.join(x['relative_path'] for x in manifest))
zip_path=UPLOAD/'v9_0a_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.rglob('*'):
        if p.is_file() and p!=zip_path: z.write(p,p.relative_to(UPLOAD))
print('Upload bundle:',zip_path)


## Known integration question

The notebook intentionally stops if the Notebook 57R1 `release_candidate` does not expose one unambiguous page-level entry point compatible with the supported adapter contract. That is the one remaining item that must come from the actual release artefacts; it is not safe to guess from version labels or reconstruct V8.8 by memory.
